In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.data_generation import constant_dgp, heterogeneous_dgp
from src.estimation import CausalTreeEvaluation
from src.simulation import SimulationRunner
from src.orchestrator import SimulationOrchestrator
from src.scenarios import scenarios
import numpy as np
pd.options.plotting.backend = "plotly"
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
np.set_printoptions(threshold=sys.maxsize)
pd.set_option('display.float_format', '{:.4f}'.format)
SIM_OUTPUT_DIR = Path(r"C:\Users\flori\Data Master_Thesis\sim_data")

In [ ]:
if __name__ == "__main__":
    orchestrator = SimulationOrchestrator(scenarios)
    results_df = orchestrator.run_all()

    results_df.to_pickle(SIM_OUTPUT_DIR / "25000_simulation_results_1000_depth_1_10_adaptive.pkl")

In [ ]:
# the code below will be at time implemetend into the general workflow

In [ ]:
results_df_adaptive = pd.read_pickle(SIM_OUTPUT_DIR/"25000_simulation_results_1000_depth_1_10_adaptive.pkl")

results_df_honest = pd.read_pickle(SIM_OUTPUT_DIR/"25000_simulation_results_1000_depth_1_10_honest.pkl")

results_df_combined = pd.concat([results_df_adaptive, results_df_honest], ignore_index=True)

In [ ]:
results_df_combined["bin_interval"] = pd.cut(results_df_combined['x_leaf_mean'], bins=200)

target_depths = list(range(1, 6))
# target_depths = list(range(6, 11))

depth_means = results_df_combined.groupby("depth")["coverage"].mean()

results_df_combined["bin_center"] = results_df_combined["bin_interval"].apply(lambda x: x.mid)

plot_df = results_df_combined.groupby(["splitting_type", "depth", "bin_center"], observed=True)["coverage"].mean().reset_index()

subplot_titles = []
for d in target_depths:
    subplot_titles.append(f'Depth = {d} (Adaptive)')
    subplot_titles.append(f'Depth = {d} (Honest)')

cover_plot = make_subplots(
    rows=len(target_depths), cols=2,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

for i, depth in enumerate(target_depths):
    depth_data = plot_df[plot_df['depth'] == depth]

    adaptive_data = depth_data[depth_data["splitting_type"] == "adaptive"].sort_values("bin_center")
    honest_data = depth_data[depth_data["splitting_type"] == "honest"].sort_values("bin_center")

    row = i + 1
    
    cover_plot.add_trace(
        go.Scatter(
            x=adaptive_data['bin_center'], 
            y=adaptive_data['coverage'], 
            mode='lines',
            name='adaptive',
            line=dict(color='red'),
            showlegend=False
        ),
        row=row, col=1
    )

    cover_plot.add_hline(
        y=adaptive_data['coverage'].mean(),
        line_color="green",
        line_dash="solid",
        row=row, col=1
    )

    # Honest goes in Column 2
    cover_plot.add_trace(
        go.Scatter(
            x=honest_data['bin_center'], 
            y=honest_data['coverage'], 
            mode='lines',
            name='honest',
            line=dict(color='blue'),
            showlegend=False
        ),
        row=row, col=2
    )

    cover_plot.add_hline(
        y=honest_data['coverage'].mean(),
        line_color="green",
        line_dash="solid",
        row=row, col=2
    )
    
    # Apply x-axis titles to both columns
    cover_plot.update_xaxes(title_text="leaf mean", row=row, col=1)
    cover_plot.update_xaxes(title_text="leaf mean", row=row, col=2)

cover_plot.update_layout(
    height=1500,
    width=1000,
    title_text="Coverage depending on leaf mean by depth (bin width = 0.5)"
)

cover_plot.update_xaxes(range=[0,100])

cover_plot.update_yaxes(
    tickvals=[0.75, 0.8, 0.85, 0.9, 0.95, 1], 
    range=[0.75, 1]
)

cover_plot.show()

In [ ]:

results_df_combined["abs_dist_from_center"] = np.abs(results_df_combined["x_leaf_mean"] - 50)

results_df_combined_extreme = results_df_combined[results_df_combined['abs_dist_from_center'] >= 45]

results_df_combined_extreme["abs_dist_center_int"] = pd.cut(results_df_combined_extreme['abs_dist_from_center'], bins=50)

target_depths = list(range(1, 6))
#target_depths = list(range(6, 11))

depth_means = results_df_combined_extreme.groupby("depth")["coverage"].mean()

results_df_combined_extreme["abs_dist_center_mid"] = results_df_combined_extreme["abs_dist_center_int"].apply(lambda x: x.mid).astype(float)

plot_df = results_df_combined_extreme.groupby(["splitting_type", "depth", "abs_dist_center_mid"], observed=True)["coverage"].mean().reset_index()

subplot_titles = []
for d in target_depths:
    subplot_titles.append(f'Depth = {d} (Adaptive)')
    subplot_titles.append(f'Depth = {d} (Honest)')

cover_plot = make_subplots(
    rows=len(target_depths), cols=2,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

for i, depth in enumerate(target_depths):
    depth_data = plot_df[plot_df['depth'] == depth]

    adaptive_data = depth_data[depth_data["splitting_type"] == "adaptive"].sort_values("abs_dist_center_mid")
    honest_data = depth_data[depth_data["splitting_type"] == "honest"].sort_values("abs_dist_center_mid")

    # Each depth gets its own row (1-indexed)
    row = i + 1
    
    # Adaptive goes in Column 1
    cover_plot.add_trace(
        go.Scatter(
            x=adaptive_data['abs_dist_center_mid'], 
            y=adaptive_data['coverage'], 
            mode='lines',
            name='adaptive',
            line=dict(color='red'),
            showlegend=False # Legend isn't strictly needed anymore since titles state the type
        ),
        row=row, col=1
    )

    cover_plot.add_hline(
        y=adaptive_data['coverage'].mean(),
        line_color="green",
        line_dash="solid",
        row=row, col=1
    )

    # Honest goes in Column 2
    cover_plot.add_trace(
        go.Scatter(
            x=honest_data['abs_dist_center_mid'], 
            y=honest_data['coverage'], 
            mode='lines',
            name='honest',
            line=dict(color='blue'),
            showlegend=False
        ),
        row=row, col=2
    )

    cover_plot.add_hline(
        y=honest_data['coverage'].mean(),
        line_color="green",
        line_dash="solid",
        row=row, col=2
    )
    
    # Apply x-axis titles to both columns
    cover_plot.update_xaxes(title_text="Abs. deviation from population mean", row=row, col=1)
    cover_plot.update_xaxes(title_text="Abs. deviation from population mean", row=row, col=2)

cover_plot.update_layout(
    height=1500,
    width=1000,
    title_text="Coverage depending on absolute distance from population mean by depth (bin width = 0.1)"
)

cover_plot.update_xaxes(range=[45, 50])
cover_plot.update_yaxes(
    tickvals=[0.75, 0.8, 0.85, 0.9, 0.95, 1], 
    range=[0.75, 1]
)

cover_plot.show()